# HW4 – Hexapawn vs AlphaBeta

สร้าง class เกม **Hexapawn** ([wikipedia](https://en.wikipedia.org/wiki/Hexapawn)) โดย subclass จาก `Game` แบบเดียวกับที่ `TicTacToe` ทำใน `games.py` แล้วให้เล่นกับ AI ที่ใช้ `alpha_beta_player`

**กฎของ Hexapawn** (กระดาน 3x3):
- `'W'` เริ่มที่แถวล่าง เดินขึ้น (+y)
- `'B'` เริ่มที่แถวบน เดินลง (-y)
- แต้มเดินตรงไปข้างหน้า 1 ช่องถ้าช่องนั้น**ว่าง** หรือกินทแยง 1 ช่องถ้ามี**หมากศัตรู**อยู่ (ไม่มีเดินสองช่องแรก ไม่มี en-passant แบบหมากรุก)
- ผู้เล่นชนะเมื่อ:
  1. เดินหมากไปถึงแถวบ้านของฝั่งตรงข้าม (breakthrough) หรือ
  2. กินหมากฝั่งตรงข้ามหมดทุกตัว หรือ
  3. ฝั่งตรงข้ามไม่มีตาเดินที่ถูกกฎหมายเหลืออยู่


In [1]:
from games import *
# ถ้าใช้ไฟล์รุ่น 4th edition แทน ให้ comment บรรทัดบนแล้วใช้บรรทัดนี้:
# from games4e import *

from collections import namedtuple

HexapawnState = namedtuple('HexapawnState', 'to_move, utility, board, moves')

## class `Hexapawn`

state เก็บ `board` เป็น dict `{(x, y): 'W' หรือ 'B'}` และ `moves` เป็น list ของการเดินที่ถูกกฎหมายของ `to_move` ที่คำนวณไว้ล่วงหน้าแล้ว (แบบเดียวกับที่ `TicTacToe` เก็บ `moves` เป็นช่องว่างที่เหลือ) — `actions()` จึงเป็นแค่การ lookup

In [2]:
class Hexapawn(Game):
    """Hexapawn บนกระดาน h x v (ปกติ 3x3)"""

    def __init__(self, h=3, v=3):
        self.h = h  # จำนวนคอลัมน์
        self.v = v  # จำนวนแถว
        board = {}
        for x in range(1, h + 1):
            board[(x, 1)] = 'W'
            board[(x, v)] = 'B'
        self.initial = HexapawnState(to_move='W', utility=0, board=board,
                                      moves=self.compute_moves(board, 'W'))

    def direction(self, player):
        """'W' เดินขึ้น (+y), 'B' เดินลง (-y)"""
        return 1 if player == 'W' else -1

    def opponent(self, player):
        return 'B' if player == 'W' else 'W'

    def compute_moves(self, board, player):
        """หาตาเดินที่ถูกกฎหมายทั้งหมดของ player บน board"""
        d = self.direction(player)
        opp = self.opponent(player)
        moves = []
        for (x, y), p in board.items():
            if p != player:
                continue
            # เดินตรงเข้าช่องว่าง
            fwd = (x, y + d)
            if 1 <= fwd[1] <= self.v and fwd not in board:
                moves.append(((x, y), fwd))
            # กินทแยง
            for dx in (-1, 1):
                diag = (x + dx, y + d)
                if 1 <= diag[0] <= self.h and 1 <= diag[1] <= self.v and board.get(diag) == opp:
                    moves.append(((x, y), diag))
        return moves

    def actions(self, state):
        return state.moves

    def result(self, state, move):
        if move not in state.moves:
            return state  # ตาเดินผิดกฎ ไม่มีผลอะไร

        (fx, fy), (tx, ty) = move
        player = state.to_move
        opponent = self.opponent(player)

        board = state.board.copy()
        del board[(fx, fy)]
        board[(tx, ty)] = player

        util = self.compute_utility(board, (tx, ty), player)
        next_moves = [] if util != 0 else self.compute_moves(board, opponent)
        if util == 0 and not next_moves:
            # ฝั่งตรงข้ามไม่มีตาเดิน -> ผู้ที่เดินล่าสุดชนะ
            util = 1 if player == 'W' else -1

        return HexapawnState(to_move=opponent, utility=util, board=board, moves=next_moves)

    def compute_utility(self, board, move_to, player):
        """+1 ถ้า 'W' เพิ่งชนะ, -1 ถ้า 'B' เพิ่งชนะ, 0 ถ้ายังไม่จบ"""
        x, y = move_to
        goal_row = self.v if player == 'W' else 1
        if y == goal_row:  # breakthrough
            return 1 if player == 'W' else -1
        opponent = self.opponent(player)
        if not any(p == opponent for p in board.values()):  # กินหมดทุกตัว
            return 1 if player == 'W' else -1
        return 0

    def utility(self, state, player):
        return state.utility if player == 'W' else -state.utility

    def terminal_test(self, state):
        return state.utility != 0 or len(state.moves) == 0

    def display(self, state):
        board = state.board
        for y in range(self.v, 0, -1):
            print(' '.join(board.get((x, y), '.') for x in range(1, self.h + 1)))
        print()

## `alpha_beta_player` — ใช้ตัวที่มีอยู่แล้วได้เลย

`games.py` / `games4e.py` มี wrapper นี้อยู่แล้ว:

```python
def alpha_beta_player(game, state):
    return alpha_beta_search(state, game)
```

เพราะ `Hexapawn` เขียน `actions` / `result` / `utility` / `terminal_test` / `to_move` ด้วยรูปแบบเดียวกับ `TicTacToe` จึงใช้ `alpha_beta_player` ตัวเดิมได้ทันที ไม่ต้องเขียนใหม่

In [3]:
print('=== alpha-beta (W) vs random (B) จำนวน 20 เกม ===')
w_wins = 0
for _ in range(20):
    game = Hexapawn()
    outcome = game.play_game(alpha_beta_player, random_player)  # W เดินก่อน, B ตาม
    w_wins += (outcome == 1)
print(f'W (alpha-beta) ชนะ {w_wins}/20 เกม ที่เล่นกับ B แบบสุ่ม')

=== alpha-beta (W) vs random (B) จำนวน 20 เกม ===
B W .
B . B
. . W

B W .
B . B
. . W

B W .
B . B
. . W

W B .
. B B
. . W

B . B
W B W
. W .

B . B
W B W
. W .

B . B
W B W
. W .

B W .
B . B
. . W

W B .
. B B
. . W

B W .
B . B
. . W

B W .
B . B
. . W

B . B
W B W
. W .

B W .
B . B
. . W

B . B
W B W
. W .

B . .
B . W
. B .

W B .
. B B
. . W

B . B
W B W
. W .

B . B
W B W
. W .

B . B
W B W
. W .

B . B
W B W
. W .

W (alpha-beta) ชนะ 19/20 เกม ที่เล่นกับ B แบบสุ่ม


In [4]:
print('=== alpha-beta (W) vs alpha-beta (B) ทั้งสองฝั่งเล่นสมบูรณ์แบบ ===')
game = Hexapawn()
outcome = game.play_game(alpha_beta_player, alpha_beta_player)
print('ผลลัพธ์สำหรับ W:', outcome,
      '(คาดว่าจะเป็น -1: บนกระดาน 3x3 ผู้เล่นคนที่สองมีวิธีบังคับชนะ)')

=== alpha-beta (W) vs alpha-beta (B) ทั้งสองฝั่งเล่นสมบูรณ์แบบ ===
. . B
. B W
B . .

ผลลัพธ์สำหรับ W: -1 (คาดว่าจะเป็น -1: บนกระดาน 3x3 ผู้เล่นคนที่สองมีวิธีบังคับชนะ)


## เล่นแบบอินเตอร์แอคทีฟ

คลิกที่ตัวหมาก `W` (สีน้ำเงิน) ของคุณเพื่อ**เลือก** (จะกลายเป็นสีเหลือง) ช่องที่เดินได้จะกลายเป็นสีเขียว จากนั้นคลิกช่องปลายทางเพื่อเดิน คลิกตัวเดิมอีกครั้งเพื่อยกเลิกการเลือก — AI (B, สีแดง) จะเดินตอบให้อัตโนมัติด้วย `alpha_beta_player`

In [6]:
import ipywidgets as widgets
from IPython.display import display

PAWN = {'W': '\u2659', 'B': '\u265F'}  # \u2659 = โ™— (คนขาว), \u265F = โ™Ÿ (คนดำ)


class InteractiveHexapawnAI:
    """เล่น Hexapawn: คุณคือ 'W', AI คือ 'B' โดยใช้ alpha_beta_player"""

    def __init__(self, h=3, v=3):
        self.game = Hexapawn(h=h, v=v)
        self.game_state = self.game.initial
        self.selected = None
        self.buttons = []
        self.status_widget = None
        self.game_container = None
        self.setup_game()

    def setup_game(self):
        h, v = self.game.h, self.game.v
        self.buttons = []
        for i in range(v):
            row = []
            for j in range(h):
                btn = widgets.Button(
                    description='',
                    layout=widgets.Layout(width='70px', height='70px'),
                    style=widgets.ButtonStyle(font_size='28px', font_weight='bold')
                )
                btn.bx = j + 1
                btn.by = v - i
                btn.on_click(self.on_button_click)
                row.append(btn)
            self.buttons.append(row)

        self.status_widget = widgets.HTML(
            value="<h3 style=\'color: blue;\'>คิวของคุณ (W) - เลือกตัวหมากที่จะเดิน</h3>"
        )

        reset_btn = widgets.Button(
            description='เริ่มเกมใหม่',
            button_style='success',
            layout=widgets.Layout(width='150px')
        )
        reset_btn.on_click(self.reset_game)

        board_rows = [widgets.HBox(row) for row in self.buttons]

        self.game_container = widgets.VBox([
            widgets.HTML("<h2 style=\'text-align: center;\'>Hexapawn vs AlphaBeta AI</h2>"),
            widgets.HTML("<p style=\'text-align: center;\'>คุณ: W (น้ำเงิน) | AI: B (แดง)</p>"),
            self.status_widget,
            widgets.VBox(board_rows, layout=widgets.Layout(align_items='center')),
            widgets.HTML("<br>"),
            widgets.HBox([reset_btn], layout=widgets.Layout(justify_content='center'))
        ])

        self.update_board_display()

    def on_button_click(self, button):
        if self.game_state.to_move != 'W':
            return

        pos = (button.bx, button.by)
        board = self.game_state.board

        if self.selected is None:
            if board.get(pos) == 'W':
                self.selected = pos
                self.update_board_display()
            return

        if pos == self.selected:
            self.selected = None
            self.update_board_display()
            return

        move = (self.selected, pos)
        if move in self.game_state.moves:
            self.game_state = self.game.result(self.game_state, move)
            self.selected = None
            self.update_board_display()

            if self.game.terminal_test(self.game_state):
                self.handle_game_over()
                return

            self.status_widget.value = "<h3 style=\'color: orange;\'>AI กำลังคิด...</h3>"
            ai_move = alpha_beta_player(self.game, self.game_state)
            self.game_state = self.game.result(self.game_state, ai_move)
            self.update_board_display()

            if self.game.terminal_test(self.game_state):
                self.handle_game_over()
            else:
                self.status_widget.value = "<h3 style=\'color: blue;\'>คิวของคุณ (W)</h3>"
        elif board.get(pos) == 'W':
            self.selected = pos
            self.update_board_display()
        # ถ้าปลายทางไม่ถูกกฎ ก็ไม่ทำอะไร

    def update_board_display(self):
        board = self.game_state.board
        legal_targets = set()
        if self.selected is not None:
            legal_targets = {m[1] for m in self.game_state.moves if m[0] == self.selected}

        for row in self.buttons:
            for btn in row:
                pos = (btn.bx, btn.by)
                player = board.get(pos)
                btn.description = PAWN.get(player, '')
                if pos == self.selected:
                    btn.button_style = 'warning'
                elif pos in legal_targets:
                    btn.button_style = 'success'
                elif player == 'W':
                    btn.button_style = 'info'
                elif player == 'B':
                    btn.button_style = 'danger'
                else:
                    btn.button_style = ''
                btn.disabled = False

    def handle_game_over(self):
        utility = self.game.utility(self.game_state, 'W')
        if utility == 1:
            self.status_widget.value = "<h3 style=\'color: green;\'>คุณชนะ! เก่งมาก!</h3>"
        elif utility == -1:
            self.status_widget.value = "<h3 style=\'color: red;\'>AI ชนะ! ลองใหม่นะ</h3>"
        else:
            self.status_widget.value = "<h3 style=\'color: purple;\'>เกมจบ (เสมอ)</h3>"

        for row in self.buttons:
            for btn in row:
                btn.disabled = True

    def reset_game(self, button=None):
        self.game_state = self.game.initial
        self.selected = None
        self.update_board_display()
        self.status_widget.value = "<h3 style=\'color: blue;\'>คิวของคุณ (W) - เลือกตัวหมากที่จะเดิน</h3>"

    def display(self):
        return self.game_container


hexapawn_app = InteractiveHexapawnAI()
display(hexapawn_app.display())